# 🌍 Análisis de Emisiones de CO2 en Chile

Este notebook analiza las emisiones de CO2 en Chile, incluyendo visualizaciones interactivas y un mapa de distribución regional de emisiones.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import folium
from pathlib import Path
import json

# Configurar rutas
DATA_DIR = Path.cwd().parent / 'app' / 'data' / 'cache'
STATIC_DIR = Path.cwd().parent / 'app' / 'static' / 'maps'

# Asegurar que los directorios existen
DATA_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Datos simulados
años = list(range(2010, 2024))
emisiones = [70.5, 72.3, 74.8, 76.1, 77.5, 79.2, 80.6, 81.8, 82.5, 83.6, 79.8, 80.5, 83.7, 85.2]

df_emisiones = pd.DataFrame({
    'Año': años,
    'Emisiones_CO2_Mt': emisiones
})

# Datos regionales
regiones_data = {
    "Metropolitana": {"lat": -33.4489, "lon": -70.6693, "emisiones": 45.2},
    "Valparaíso": {"lat": -33.0458, "lon": -71.6197, "emisiones": 12.8},
    "Biobío": {"lat": -36.8201, "lon": -73.0443, "emisiones": 18.5},
    "Antofagasta": {"lat": -23.6509, "lon": -70.3975, "emisiones": 25.3},
    "O'Higgins": {"lat": -34.1708, "lon": -70.7444, "emisiones": 15.6},
    "Maule": {"lat": -35.4264, "lon": -71.6553, "emisiones": 8.9},
    "Araucanía": {"lat": -38.7359, "lon": -72.5904, "emisiones": 7.4},
    "Los Lagos": {"lat": -41.4693, "lon": -72.9424, "emisiones": 6.8}
}

# Guardar datos para la aplicación
emisiones_anual = df_emisiones.set_index('Año').to_dict()['Emisiones_CO2_Mt']

with open(DATA_DIR / "emisiones_anuales.json", 'w') as f:
    json.dump(emisiones_anual, f)

with open(DATA_DIR / "emisiones_regionales.json", 'w') as f:
    json.dump(regiones_data, f)

## 📈 Tendencia Histórica de Emisiones

In [ ]:
# Gráfico de tendencia
fig = px.line(
    df_emisiones, 
    x='Año', 
    y='Emisiones_CO2_Mt',
    title='Emisiones de CO2 en Chile (2010-2024)',
    markers=True
)
fig.show()

## 🗺️ Distribución Regional de Emisiones

In [ ]:
# Crear mapa base
m = folium.Map(
    location=[-33.4489, -70.6693],
    zoom_start=5,
    tiles='cartodbpositron'
)

# Añadir marcadores de emisiones
for region, data in regiones_data.items():
    # Color basado en nivel de emisiones
    color = 'red' if data['emisiones'] > 20 else 'orange' if data['emisiones'] > 10 else 'green'
    
    folium.CircleMarker(
        location=[data['lat'], data['lon']],
        radius=data['emisiones']/2,  # Tamaño proporcional a emisiones
        popup=f"{region}<br>Emisiones: {data['emisiones']} Mt CO2",
        color=color,
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

# Añadir leyenda
legend_html = """
<div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid grey; border-radius: 5px;">
    <h4>Emisiones de CO2 (Mt)</h4>
    <p><i style="background: red; width: 10px; height: 10px; display: inline-block;"></i> > 20 Mt</p>
    <p><i style="background: orange; width: 10px; height: 10px; display: inline-block;"></i> 10-20 Mt</p>
    <p><i style="background: green; width: 10px; height: 10px; display: inline-block;"></i> < 10 Mt</p>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Guardar mapa
m.save(str(STATIC_DIR / "emisiones_co2_latest.html"))

# Mostrar mapa
m

## 📊 Tabla de Emisiones por Región

In [ ]:
# Crear DataFrame de emisiones regionales
df_regional = pd.DataFrame(regiones_data).T.reset_index()
df_regional.columns = ['Region', 'lat', 'lon', 'emisiones']

# Mostrar tabla ordenada por emisiones
df_regional[['Region', 'emisiones']].sort_values('emisiones', ascending=False)